In [ ]:

from collections import Counter
import pandas as pd 
import re
import requests
import json
import time
import itertools
from itertools import islice
import glob
import pprint

import pylab as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.colors import ListedColormap
pp = pprint.PrettyPrinter()
import numpy as np

import regex as re

import os
import sys
import ast
from datetime import datetime
import glob

API_KEY = require_env("LENS_API_KEY")



In [ ]:
#adjust the function so the re-merging of file names works better (dict to keep track of original and edited doi names) 

In [ ]:
# Resolve the repository and external data directory without machine-specific paths.
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "project_config.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("Run this notebook from within the cloned repository.")

from project_config import REPO_ROOT, DATA_ROOT, require_env

from pathlib import Path
import sys

# Code is run from the OneDrive project code folder.
CODE_ROOT = REPO_ROOT
DATASET_CURATION_CODE_ROOT = CODE_ROOT / "dataset_curation"
FUNCTIONS_DIR = DATASET_CURATION_CODE_ROOT / "functions"
if not FUNCTIONS_DIR.exists():
    raise FileNotFoundError(
        f"Functions folder not found: {FUNCTIONS_DIR}. "
        "Copy code/dataset_curation/functions to the OneDrive code root or adjust CODE_ROOT."
    )

if str(FUNCTIONS_DIR) not in sys.path:
    sys.path.insert(0, str(FUNCTIONS_DIR))

# Data is read from and written to the share-drive project folder.
SHARE_ROOT = DATA_ROOT
# DATA_ROOT is configured through project_config.

# Keep legacy string-style path variable because the downstream cells use
# expressions like path + r"processed_data\...".
path = str(DATA_ROOT) + "\\"

from download_oa_pdfs import download_pdfs_for_publications
from download_oa_pdfs import download_pdfs_by_doi


In [ ]:
df_scholarly = pd.read_csv(path+R'processed_data\merged_all.csv', index_col = 0)

In [ ]:
Counter(df_scholarly['title']).most_common()

In [ ]:
df_scholarly_clean = df_scholarly.drop_duplicates(subset=['title', 'abstract','dates'])

In [ ]:
fields_of_study_clean = []
for fields in df_scholarly_clean['fields_of_study'].dropna().to_list():
    for field in ast.literal_eval(fields): 
        fields_of_study_clean.append(field)
    

In [ ]:
asjc_subjects_clean = []
for fields in df_scholarly_clean['asjc_subjects'].dropna().to_list():
    for field in ast.literal_eval(fields): 
        asjc_subjects_clean.append(field)
  

In [ ]:
#save subject list to mark things to inspect
pd.DataFrame(list(set(asjc_subjects_clean)), columns = ['subjects']).to_csv(path+r'processed_data\subject_list.csv')

### create subject exclusion conditions

In [ ]:
df_inspect = pd.read_csv(path+r'processed_data\subject_list.csv', index_col = 0)

In [ ]:
non_inspect_list = df_inspect[df_inspect['inspect'] == 1]['subjects'].to_list()

In [ ]:
len(non_inspect_list)

In [ ]:
subject_list_to_inspect = []
subject_list_fine = []
for subject_list in df_scholarly_clean['asjc_subjects'].dropna().to_list(): 
    inspect_subject_list = 0
    for subject in ast.literal_eval(subject_list):
        
        if subject in non_inspect_list:
            
            
            inspect_subject_list = 1
    
    if inspect_subject_list == 1: 
        subject_list_fine.append(subject_list)
    else: 
        subject_list_to_inspect.append(subject_list)
        

In [ ]:
len(set(subject_list_fine))

In [ ]:
len(set(subject_list_to_inspect))

In [ ]:
df_inspect = df_scholarly_clean[df_scholarly_clean['asjc_subjects'].isin(subject_list_to_inspect)]

In [ ]:
inspect_indiv = []
for subject in df_scholarly_clean['asjc_subjects'].dropna().to_list(): 
    if 'Developmental Biology' in subject: 
        inspect_indiv.append(subject)

In [ ]:
df_scholarly_clean[df_scholarly_clean['asjc_subjects'].isin(inspect_indiv)]

In [ ]:
df_scholarly_clean[df_scholarly_clean['asjc_subjects'].isin(inspect_indiv)]['title'].to_list()

In [ ]:
df_scholarly_clean[df_scholarly_clean['asjc_subjects'].isin(inspect_indiv)]['asjc_subjects'].to_list()

In [ ]:
df_inspect[df_inspect['asjc_subjects'].isin(inspect_indiv)]['fields_of_study'].to_list()

In [ ]:
df_scholarly_clean[df_scholarly_clean['asjc_subjects'].isin(subject_list_fine)][['title','abstract','asjc_subjects', 'fields_of_study', 'dates','doi','open_access_pdf_urls']].reset_index(drop = True).to_excel(path+r'processed_data\scholarly_final_reduced.xlsx', index=False)